In [1]:
 import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
 
# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [18]:
train_df = pd.read_csv("sp500_train (2).csv")
test_df  = pd.read_csv("sp500_test (2).csv")

train_df["Date"] = pd.to_datetime(train_df["Date"])
test_df["Date"]  = pd.to_datetime(test_df["Date"])

train_df = train_df.sort_values("Date").set_index("Date")
test_df  = test_df.sort_values("Date").set_index("Date")

In [19]:
# =============================================================================
# 2. WALK-FORWARD DATASET CONSTRUCTION
# =============================================================================
# Paper: "four previous days of true lagged volatility" as features,
#        walk-forward validation scheme to prevent data leakage.
#
#   - Input  X[i] : vol[i-4], vol[i-3], vol[i-2], vol[i-1]  (4 lagged values)
#   - Target y[i] : vol[i]  (next-day realised vol)
#
# CRITICAL (paper section III-C): build sequences from each split independently
# so no train-boundary rows bleed into test sequences.
 
LOOKBACK  = 4   # number of lagged vol values fed to LSTM (paper III-C)
PRED_STEP = 1   # predict 1 step ahead
 
def build_sequences(series, lookback=4):
    """Convert a 1-D volatility array into (X, y) LSTM input sequences."""
    X, y = [], []
    for i in range(lookback, len(series) - PRED_STEP + 1):
        X.append(series[i - lookback : i])   # shape: (lookback,)
        y.append(series[i])                  # scalar: next vol
    return np.array(X), np.array(y)
 
X_train_raw, y_train_raw = build_sequences(train_df["Volatility"].values, lookback=LOOKBACK)
X_test_raw,  y_test_raw  = build_sequences(test_df["Volatility"].values,  lookback=LOOKBACK)
 
# Scaling — fit on train only (paper III-C: no leakage)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
 
X_train_scaled = scaler_X.fit_transform(X_train_raw)
X_test_scaled  = scaler_X.transform(X_test_raw)
 
y_train_scaled = scaler_y.fit_transform(y_train_raw.reshape(-1, 1)).ravel()
y_test_scaled  = scaler_y.transform(y_test_raw.reshape(-1, 1)).ravel()
 
# Reshape X for LSTM: (samples, timesteps, features)
X_train = X_train_scaled.reshape(-1, LOOKBACK, 1)
X_test  = X_test_scaled.reshape(-1, LOOKBACK, 1)
 
print(f"\nTrain set : X={X_train.shape}  y={y_train_scaled.shape}")
print(f"Test  set : X={X_test.shape}   y={y_test_scaled.shape}")
 


Train set : X=(4001, 4, 1)  y=(4001,)
Test  set : X=(998, 4, 1)   y=(998,)


In [20]:
# =============================================================================
# 3. MODEL ARCHITECTURE  (paper III-C)
# =============================================================================
# - One LSTM layer with 64 neurons
# - Dropout rate: 0.2
# - One Dense output layer (single scalar prediction)
 
model = Sequential([
    LSTM(64, input_shape=(LOOKBACK, 1), return_sequences=False),
    Dropout(0.2),
    Dense(1)
], name="LSTM_Vol_Forecaster")
 
model.compile(optimizer="adam", loss="mse")
model.summary()

C:\Users\fairy\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "LSTM_Vol_Forecaster"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,961 (66.25 KB)

 Trainable params: 16,961 (66.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# =============================================================================
# 4. TRAINING
# =============================================================================
# EarlyStopping for regularisation (patience=20, restore best weights).
# Paper does not specify exact epochs/batch; these are sensible defaults.
 
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)
 
history = model.fit(
    X_train, y_train_scaled,
    epochs=200,
    batch_size=32,
    validation_split=0.10,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.0040 - val_loss: 3.4954e-04
Epoch 2/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 8.4979e-04 - val_loss: 3.4657e-04
Epoch 3/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 7.4018e-04 - val_loss: 3.9829e-04
Epoch 4/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 8.2737e-04 - val_loss: 4.0356e-04
Epoch 5/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 6.5968e-04 - val_loss: 3.9484e-04
Epoch 6/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 6.5662e-04 - val_loss: 3.5693e-04
Epoch 7/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 6.5407e-04 - val_loss: 3.2148e-04
Epoch 8/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 6.0513e-04 - val_loss: 3.2860e-04
Epoch 9/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5.5621e-04 - val_loss: 2.9768e-04
Epoch 10/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5.3010e-04 - val_loss: 3.1448e-04
Epoch 11/200
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/ste